# 🫀 Risk Factors for Hypertension — Part 3A
## Case-Control Study Design, Chi-Square Tests & Crude Odds Ratios
### Epidemiology with Python | Tutorial Series by Desy Nuryunarsih

---
**Study background:**  
Hypertension is one of the most common non-communicable diseases in Indonesia,  
affecting nearly 1 in 3 adults and a major driver of stroke and heart disease.

In this tutorial, we use a **simulated case-control dataset** to explore environmental  
and lifestyle risk factors for hypertension in an Indonesian urban community.

**In Part 3A, you will learn to:**
- Understand case-control study design
- Create and explore a simulated dataset
- Build 2×2 contingency tables
- Run chi-square tests
- Calculate crude Odds Ratios (OR) with 95% confidence intervals
- Interpret results correctly

**In Part 3B** (next video), we run multivariate logistic regression and build a forest plot.


In [ ]:
# ── Install (run once if needed) ──────────────────────
# !pip install pandas numpy scipy matplotlib seaborn

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")


## 📚 Step 1: Case-Control Study Design

In a **case-control study**:
- **Cases** = adults diagnosed with hypertension
- **Controls** = adults without hypertension, matched by age group
- We look **backwards** to measure past exposures
- We measure association using the **Odds Ratio (OR)**

### Why Odds Ratio and not Risk Ratio?
In case-control studies we **select people based on their outcome**, not their exposure.  
This means we cannot directly estimate risk in the population — so we use OR instead.

| | Hypertension (Case) | No Hypertension (Control) |
|---|---|---|
| **Exposed** | a | b |
| **Not Exposed** | c | d |

**OR = (a × d) / (b × c)**

- OR > 1 → exposure increases the **odds** of hypertension (risk factor)
- OR < 1 → exposure **reduces** the odds (protective factor)
- OR = 1 → no association


In [ ]:
# ══════════════════════════════════════════════════════
# Step 2: Create Simulated Dataset
# ══════════════════════════════════════════════════════
# N = 200 (100 cases with hypertension, 100 controls)
# Risk factors are inspired by epidemiological literature
# All data are SIMULATED for educational purposes only.

np.random.seed(2024)
n_case = 250
n_ctrl = 250

outcome = [1] * n_case + [0] * n_ctrl  # 1 = hypertension

def sim(p_case, p_ctrl, seed=0):
    np.random.seed(2024 + seed)
    return list(np.random.binomial(1, p_case, n_case)) +            list(np.random.binomial(1, p_ctrl,  n_ctrl))

data = {
    'hypertension':       outcome,
    # Risk factors
    'high_salt_diet':     sim(0.72, 0.35, 1),   # strong risk
    'physically_inactive':sim(0.65, 0.32, 2),   # strong risk
    'overweight_obese':   sim(0.60, 0.28, 3),   # strong risk
    'family_history':     sim(0.55, 0.25, 4),   # strong risk
    'high_stress':        sim(0.50, 0.30, 5),   # moderate risk
    'smoker':             sim(0.42, 0.28, 6),   # moderate risk
    'alcohol_use':        sim(0.30, 0.22, 7),   # weak / non-sig
    'low_education':      sim(0.35, 0.30, 8),   # non-sig
    # Protective factors
    'fruit_veg_daily':    sim(0.25, 0.50, 9),   # protective
    'regular_checkup':    sim(0.20, 0.45, 10),  # protective
}

df = pd.DataFrame(data)
df['label'] = df['hypertension'].map({1: 'Hypertension', 0: 'Control'})

print("Dataset shape:", df.shape)
print("\nOutcome distribution:")
print(df['label'].value_counts())
df.head()


In [ ]:
# ══════════════════════════════════════════════════════
# Step 3: Explore the Dataset
# ══════════════════════════════════════════════════════

risk_factors = [
    'high_salt_diet', 'physically_inactive', 'overweight_obese',
    'family_history', 'high_stress', 'smoker', 'alcohol_use',
    'low_education', 'fruit_veg_daily', 'regular_checkup'
]

print(f"{'Risk Factor':<25} {'Cases (%)':>12} {'Controls (%)':>14}")
print("-" * 53)
for rf in risk_factors:
    pct_case = df[df['hypertension']==1][rf].mean() * 100
    pct_ctrl = df[df['hypertension']==0][rf].mean() * 100
    print(f"{rf:<25} {pct_case:>11.1f}% {pct_ctrl:>13.1f}%")


## 📊 Step 4: Chi-Square Tests and Crude Odds Ratios

For each risk factor we will:
1. Build a **2×2 contingency table**
2. Run a **chi-square test**
3. Calculate the **crude OR** with **95% Confidence Interval**

### The 95% CI for OR uses the log method:
```
log(OR) ± 1.96 × √(1/a + 1/b + 1/c + 1/d)
```
Then exponentiate back to the OR scale.


In [ ]:
# ══════════════════════════════════════════════════════
# Helper function: chi-square + crude OR + 95% CI
# ══════════════════════════════════════════════════════

def crude_or_analysis(df, exposure, outcome='hypertension'):
    """
    Inputs:
        df       : DataFrame
        exposure : column name (1=exposed, 0=unexposed)
        outcome  : column name (1=case, 0=control)
    Returns:
        dict with chi2, p-value, OR, 95% CI, significance
    """
    table = pd.crosstab(df[exposure], df[outcome])

    # 2x2 cells
    a = table.loc[1, 1]  # exposed & case
    b = table.loc[1, 0]  # exposed & control
    c = table.loc[0, 1]  # unexposed & case
    d = table.loc[0, 0]  # unexposed & control

    # Chi-square
    chi2, p, dof, _ = chi2_contingency(table)

    # Crude OR + 95% CI (log method)
    if 0 in [a, b, c, d]:
        or_val = float('nan')
        ci_lo = ci_hi = float('nan')
    else:
        or_val = (a * d) / (b * c)
        log_or = np.log(or_val)
        se     = np.sqrt(1/a + 1/b + 1/c + 1/d)
        ci_lo  = np.exp(log_or - 1.96 * se)
        ci_hi  = np.exp(log_or + 1.96 * se)

    return {
        'exposure': exposure,
        'a': a, 'b': b, 'c': c, 'd': d,
        'chi2':   round(chi2, 3),
        'p':      round(p, 3),
        'or':     round(or_val, 2),
        'ci_lo':  round(ci_lo,  2),
        'ci_hi':  round(ci_hi,  2),
        'sig':    p < 0.05,
        'table':  table
    }

print("✅ Function defined. Ready to analyse!")


In [ ]:
# ══════════════════════════════════════════════════════
# Detailed Walkthrough: HIGH SALT DIET
# ══════════════════════════════════════════════════════

res = crude_or_analysis(df, 'high_salt_diet')

print("=== 2×2 Table: High Salt Diet vs Hypertension ===\n")
print(res['table'].rename(
    index={0: 'Low salt diet', 1: 'High salt diet'},
    columns={0: 'Control', 1: 'Hypertension'}
))

print(f"\n  a (high salt & hypertension)  = {res['a']}")
print(f"  b (high salt & control)        = {res['b']}")
print(f"  c (low salt  & hypertension)   = {res['c']}")
print(f"  d (low salt  & control)        = {res['d']}")

print(f"\n  OR = (a×d)/(b×c) = ({res['a']}×{res['d']}) / ({res['b']}×{res['c']})")
print(f"     = {res['or']}")
print(f"\n  Chi-square statistic : {res['chi2']}")
print(f"  p-value              : {res['p']}")
print(f"  95% CI               : ({res['ci_lo']} – {res['ci_hi']})")

print("\n📌 Interpretation:")
if res['sig']:
    print(f"  High salt diet is a SIGNIFICANT risk factor for hypertension.")
    print(f"  People with high salt intake have {res['or']}× higher odds of hypertension")
    print(f"  compared to those with low salt intake (p = {res['p']}).")
else:
    print("  No statistically significant association (p ≥ 0.05).")


In [ ]:
# ══════════════════════════════════════════════════════
# Step 5: Univariate Analysis — All Risk Factors
# ══════════════════════════════════════════════════════

all_results = [crude_or_analysis(df, rf) for rf in risk_factors]

summary = pd.DataFrame([{
    'Risk Factor':   r['exposure'].replace('_', ' ').title(),
    'Cases (%)':    f"{r['a'] / n_case * 100:.1f}%",
    'Controls (%)': f"{r['b'] / n_ctrl * 100:.1f}%",
    'Chi²':          r['chi2'],
    'p-value':       r['p'],
    'Crude OR':      r['or'],
    '95% CI':       f"({r['ci_lo']} – {r['ci_hi']})",
    'Sig':          '✅' if r['sig'] else '—'
} for r in all_results])

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)
print("=== Univariate Analysis: All Risk Factors ===\n")
print(summary.to_string(index=False))


In [ ]:
# ══════════════════════════════════════════════════════
# Step 6: Visualise Crude Odds Ratios
# ══════════════════════════════════════════════════════

plot_res = [r for r in all_results if not np.isnan(r['or'])]
labels  = [r['exposure'].replace('_', ' ').title() for r in plot_res]
ors     = [r['or']   for r in plot_res]
ci_lo   = [r['ci_lo'] for r in plot_res]
ci_hi   = [r['ci_hi'] for r in plot_res]
sigs    = [r['sig']  for r in plot_res]

# Sort by OR for cleaner display
order   = np.argsort(ors)
labels  = [labels[i]  for i in order]
ors     = [ors[i]     for i in order]
ci_lo   = [ci_lo[i]   for i in order]
ci_hi   = [ci_hi[i]   for i in order]
sigs    = [sigs[i]    for i in order]

colors  = ['firebrick'   if (s and o > 1) else
           'forestgreen' if (s and o < 1) else
           'steelblue'
           for o, s in zip(ors, sigs)]

yerr_lo = [o - lo for o, lo in zip(ors, ci_lo)]
yerr_hi = [hi - o for o, hi in zip(ors, ci_hi)]

fig, ax = plt.subplots(figsize=(12, 7))
y_pos   = range(len(labels))

ax.barh(y_pos, ors,
        xerr=[yerr_lo, yerr_hi],
        color=colors, alpha=0.82,
        edgecolor='black', linewidth=0.7,
        capsize=4, error_kw={'linewidth': 1.8})

ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Crude Odds Ratio (95% CI)', fontsize=12)
ax.set_title(
    'Crude Odds Ratios for Hypertension Risk Factors\n'
    'Case-Control Study, N=500 (250 cases, 250 controls)',
    fontsize=13, fontweight='bold')

legend_elements = [
    mpatches.Patch(facecolor='firebrick',   label='Significant risk factor (OR > 1)'),
    mpatches.Patch(facecolor='forestgreen', label='Significant protective factor (OR < 1)'),
    mpatches.Patch(facecolor='steelblue',   label='Not significant'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('part3a_crude_or_hypertension.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved.")


## ✅ Part 3A Summary

| Concept | Key Point |
|---|---|
| Case-control design | Select on outcome, look back at exposures |
| 2×2 contingency table | Foundation of OR calculation |
| Chi-square test | Is the association statistically significant? |
| Crude Odds Ratio | OR = (a×d)/(b×c) — unadjusted strength of association |
| 95% Confidence Interval | If it excludes 1.0 → statistically significant |
| Interpretation | OR > 1 = risk factor; OR < 1 = protective |

---

### 🔍 What we found (crude analysis):
- **High salt diet, physical inactivity, overweight, family history** → significant risk factors
- **Fruit & vegetable intake, regular health check-up** → significant protective factors
- **Alcohol use, low education** → not statistically significant

---

**🎯 In Part 3B**, we adjust for all these factors simultaneously using  
**multivariate logistic regression** and build a **forest plot**.

*Don't forget to subscribe so you don't miss it!* 🔔
